Oracle AI Data Platform v1.0

# Notion Ingestion (Full Refresh and CDC)

Lands Notion **pages**, **data sources**, **blocks** and **users** in Delta tables in the catalog and schema named in `notion_ingest.yaml`, using the `aidp-connector-notion` package.

- **CDC (default):** reads only what changed since the last successful run, using a per-object watermark kept in `notion_sync_state`.
- **Full:** reloads everything and removes what no longer exists in Notion.

**Before you run**

1. Create a Notion internal integration, copy its secret, and share the pages and databases you want with it.
2. Store the secret in the AIDP Credential Store and set `notion.credential_name` in the config.
3. Install the `aidp-connector-notion` wheel on the cluster (see step 1 below).
4. Copy `notion_ingest.sample.yaml` to `notion_ingest.yaml` in your workspace and edit the target catalog and schema.

See the package `README.md` for details and known limits.

## 1. Install the connector

Skip this cell if the wheel is already installed as a cluster library. Otherwise upload `aidp_connector_notion-<version>-py3-none-any.whl` to your workspace, then uncomment the line below and set its path. It also installs `requests` and `pyyaml`.

In [ ]:
# %pip install /Workspace/<path-to>/aidp_connector_notion-0.1.0-py3-none-any.whl

In [ ]:
from aidp_connector_notion import (
    __version__,
    format_summary,
    load_config,
    raise_on_failure,
    read_token,
    run,
)

print("aidp-connector-notion", __version__)

## 2. Load the configuration

In [ ]:
# Your copy of notion_ingest.sample.yaml.
CONFIG_PATH = "/Workspace/<path-to>/notion_ingest.yaml"

config = load_config(CONFIG_PATH)

print("target :", config.target.qualified_schema)
print("objects:", ", ".join(config.sync.objects))
print("mode   :", config.sync.mode)

## 3. Optional per-run override

When this notebook runs as a job, a job parameter `MODE` (`cdc` or `full`) overrides `sync.mode` for that run only. Interactive runs use the config.

In [ ]:
_MISSING = "\x00__NO_PARAMETER__"


def job_parameter(name):
    """Read a job parameter, or None. Names are case-sensitive on the platform,
    so try the usual spellings: whoever registers the job may write 'mode', not 'MODE'."""
    for candidate in dict.fromkeys((name, name.upper(), name.lower())):
        try:
            # oidlUtils is provided by the AIDP runtime - do not import it.
            value = oidlUtils.parameters.getParameter(candidate, _MISSING)  # noqa: F821
        except NameError:
            return None  # not running on AIDP
        if not (value == _MISSING or value is None or str(value).strip() == ""):
            return str(value).strip()
    return None


MODE = job_parameter("MODE")
print("mode override:", MODE or "(none)")

## 4. Read the Notion token

In [ ]:
try:
    get_secret = aidputils.secrets.get  # noqa: F821 - provided by the AIDP runtime, do not import it
except NameError:
    get_secret = None  # not on AIDP: read the token from the environment variable instead

token = read_token(config.notion, get_secret)
print("token read (not shown)")

## 5. Run the sync

In [ ]:
summaries = run(spark, config, token, mode_override=MODE)  # noqa: F821 - spark is provided by the notebook

print(format_summary(summaries))
raise_on_failure(summaries)

## 6. Look at the result

```sql
SELECT id, title, last_edited_time, in_trash FROM <catalog>.<schema>.pages ORDER BY last_edited_time DESC LIMIT 20;

-- Text of a page, in reading order
SELECT depth, position, type, raw_json FROM <catalog>.<schema>.blocks WHERE page_id = '<page id>';

-- Watermarks and the outcome of the last run per object
SELECT * FROM <catalog>.<schema>.notion_sync_state;
```